# Stage 20 · MIDI 2026 — Dataset Verification & Source Documentation (v1)

**Scope:** Documentary audit of the **original immutable 1,000-question development corpus and pre-selected 250-question final test** against a later 1,300-row manually sourced verification workbook. The extra 50 rows must remain **RESERVE**, not enter either evaluation split. No LLM inference, answer generation, model fitting, label updates or retroactive test selection.

**Research integrity:** The source workbook is *post-hoc provenance documentation*; existence of a URL is **not** an independent confirmation that the page supports every claim. `needs review` remains an open documentation issue even if a second check was described orally. No agreement statistic, second annotator, or timestamp is invented. The original JSON contains `verified` status flags, whereas the later workbook may express outstanding concerns; **both are preserved**. Source/domain frequency is **not** evidence of accuracy or prestige.

**Inputs (immutable):** `validated_v007.json` (original 1,000), `validated_v012_final250_v1.json` (original 1,250), Stage 13 final-test selection manifest, and `midi_revize_data.xlsx` (1,300 sourced records). **Outputs:** new `results/midi2026_revision/v1/04_dataset_documentation/` folder only. Existing Stage 08–19 artifacts are never written.

In [ ]:
# 1 · Configuration — Google Colab or local filesystem; no external API calls
from pathlib import Path
import os, csv, io, json, hashlib, re, zipfile, collections
import urllib.parse
import xml.etree.ElementTree as ET
from datetime import datetime, timezone

INPUT_MODE='drive'  # 'drive' in Colab, or 'local' if you uploaded source files into /content
if INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
    ORIGINAL1000=ROOT/'data'/'validated'/'validated_v007.json'
    ORIGINAL1250=ROOT/'data'/'validated'/'validated_v012_final250_v1.json'
    SELECTION_MANIFEST=ROOT/'results'/'stage13'/'validated_v012_final250_v1'/'stage13_final_test_manifest_v1.json'
    # Copy the uploaded xlsx into this directory in Drive, WITHOUT changing its content.
    SOURCED_XLSX=ROOT/'data'/'review_evidence'/'midi_revize_data.xlsx'
    OUTPUT=ROOT/'results'/'midi2026_revision'/'v1'/'04_dataset_documentation'
else:
    ROOT=Path('/content')
    ORIGINAL1000=ROOT/'validated_v007.json'
    ORIGINAL1250=ROOT/'validated_v012_final250_v1.json'
    SELECTION_MANIFEST=ROOT/'stage13_final_test_manifest_v1.json'
    SOURCED_XLSX=ROOT/'midi_revize_data.xlsx'
    OUTPUT=ROOT/'stage20_output'

EXPECTED_SHA={
    'original1000':'f169c3e78bbe73bb4e676f70433e069dada980d0b3caf01ffc348aaedc78ac8a',
    'original1250':'6c1d3866c9f7fbab89105d9a53c26cc35715b0eb6c4264a669aa10c81845b680',
    'stage13_selection':'5742d8c4df190373358a9e0533e55b57ea334ab20d87606cd3a0d320b910fc94',
    'sourced_xlsx':'3faf378f971bf966b0651411ac439a9f633828217fa0f544ae2d9dd62a16867b'
}
REVIEWERS_REPORTED_BY_RESEARCHER=1  # Based on the user's account; NOT inferred from file metadata

def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
def jread(path):
    with open(path,encoding='utf-8') as f:return json.load(f)

def cstrip(v):return '' if v is None else str(v).strip()
def as_int(v,default=0):
    try:return int(v)
    except (TypeError,ValueError):return default

print('Inputs configured. No file is modified. Outputs:',OUTPUT)

In [ ]:
# 2 · Cryptographic file identity and Stage 13 independent test membership
INPUTS={'original1000':ORIGINAL1000,'original1250':ORIGINAL1250,
        'stage13_selection':SELECTION_MANIFEST,'sourced_xlsx':SOURCED_XLSX}
ACTUAL_SHA={}
for name,path in INPUTS.items():
    require(path.is_file(),f'MISSING {name}: {path}')
    actual=sha256(path)
    require(actual==EXPECTED_SHA[name],f'SHA mismatch for {name}. Expected {EXPECTED_SHA[name]}, received {actual}. Never silently substitute versions.')
    ACTUAL_SHA[name]=actual
    print('PASS',name,actual[:16],path.name)

old=jread(ORIGINAL1000)
combined=jread(ORIGINAL1250)
select=jread(SELECTION_MANIFEST)
require(isinstance(old,list) and isinstance(combined,list),'Expected question arrays in original JSON')
require(isinstance(select,dict),'Stage13 manifest must be a JSON object')
TEST_IDS=set(select.get('final_question_ids',[]))
OLD_IDS=set(x['question_id'] for x in old)
COMBINED_IDS=set(x['question_id'] for x in combined)
require(len(OLD_IDS)==1000 and len(old)==1000,'Original dataset must contain 1,000 unique questions')
require(len(TEST_IDS)==250,'Stage13 manifest must hold exactly 250 selected test IDs')
require(len(COMBINED_IDS)==1250 and len(combined)==1250,'Final combined JSON should have 1250 unique questions')
require(not (TEST_IDS & OLD_IDS),'Stage13 selected test overlaps original 1000')
require(COMBINED_IDS==OLD_IDS|TEST_IDS,'Combined JSON must be exactly development1000 + Stage13 final250')

# Stage03 model-selection exclusions: verify from Stage14 feature matrix when necessary;
# this audit does not invent which 30 IDs were excluded from Clean-Dev.
print('Stage13 split confirmed: original1000',len(OLD_IDS),'test250',len(TEST_IDS))

In [ ]:
# 3 · Read the original Excel through standard XLSX (OpenXML) data, no editing.
# Compatible with Google Colab without pip installs or Excel/LibreOffice automation.
# Supports sharedStrings and inline strings. Workbook has one data sheet.
XNS='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'

def xlsx_first_sheet(path):
    with zipfile.ZipFile(path) as z:
        shared=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            root=ET.fromstring(z.read('xl/sharedStrings.xml'))
            for si in root.findall(f'{XNS}si'):
                shared.append(''.join(t.text or '' for t in si.iter(f'{XNS}t')))
        files=sorted((n for n in z.namelist() if re.fullmatch(r'xl/worksheets/sheet\d+\.xml',n)))
        require(bool(files),'No worksheet XML in XLSX')
        root=ET.fromstring(z.read(files[0]))
        data=[]
        for row in root.findall(f'.//{XNS}sheetData/{XNS}row'):
            vals={}
            for cell in row.findall(f'{XNS}c'):
                coord=cell.attrib.get('r','')
                match=re.match(r'[A-Z]+',coord)
                if not match:continue
                idx=0
                for ch in match.group(0):idx=idx*26+ord(ch)-ord('A')+1
                idx-=1
                typ=cell.attrib.get('t','')
                v=cell.find(f'{XNS}v')
                inline=cell.find(f'{XNS}is')
                if typ=='s':value=shared[int(v.text)] if v is not None and v.text else ''
                elif typ=='inlineStr':value=''.join(t.text or '' for t in inline.iter(f'{XNS}t')) if inline is not None else ''
                elif typ=='b':value=bool(v is not None and v.text=='1')
                elif v is None or v.text is None:value=''
                else:
                    raw=v.text
                    try:value=float(raw) if '.' in raw or 'E' in raw.upper() else int(raw)
                    except ValueError:value=raw
                vals[idx]=value
            if vals: data.append([vals.get(i,'') for i in range(max(vals)+1)])
        require(len(data)>=2,'Excel sheet has no data rows')
        hdr=[cstrip(x) for x in data[0]]
        require(len(hdr)==len(set(hdr)),'Excel header has duplicates')
        rows=[]
        for rr in data[1:]:
            r={h:(rr[i] if i<len(rr) else '') for i,h in enumerate(hdr)}
            if cstrip(r.get('question_id')):rows.append(r)
        return hdr,rows

HEADERS,EXCEL=xlsx_first_sheet(SOURCED_XLSX)
NEEDED={'question_id','record_fingerprint','question','reference_answer','accepted_aliases',
        'answer_type','category','source_name','source_reference','verification_status','validation_notes',
        'factually_correct','single_unambiguous_answer','time_stable','aliases_valid'}
require(NEEDED.issubset(HEADERS),f'Missing columns: {sorted(NEEDED-set(HEADERS))}')
EXCEL_BY={r['question_id']:r for r in EXCEL}
require(len(EXCEL_BY)==len(EXCEL)==1300,'Expected exactly 1,300 unique sourced Excel records')
RESERVE_IDS=set(EXCEL_BY)-COMBINED_IDS
require(len(RESERVE_IDS)==50 and COMBINED_IDS.issubset(EXCEL_BY),'Excel should cover all 1250 used questions and exactly 50 reserve records')
print('Excel',len(EXCEL),'rows; matched original',len(COMBINED_IDS),'reserve',len(RESERVE_IDS))

In [ ]:
# 4 · Validate unchanged question content, aliases and record fingerprints.
# The revision Excel is joined ONLY on question_id. No original question/label is updated.
BY_1250={x['question_id']:x for x in combined}
BY_1000={x['question_id']:x for x in old}
MATCH_FIELDS=['question','reference_answer','answer_type','category','record_fingerprint','accepted_aliases']
DIFFS=[]
for qid,x in BY_1250.items():
    r=EXCEL_BY[qid]
    for field in MATCH_FIELDS:
        if field=='record_fingerprint': a=x.get('validation',{}).get('record_fingerprint'); b=r.get(field)
        elif field=='accepted_aliases':a=x[field]; b=[v.strip() for v in cstrip(r[field]).split('|')]
        else:a=x.get(field);b=r.get(field)
        if a!=b:DIFFS.append({'question_id':qid,'field':field,'json_value':str(a)[:300], 'xlsx_value':str(b)[:300]})
for qid,x in BY_1000.items():
    y=BY_1250.get(qid)
    if not y or any(x.get(f)!=y.get(f) for f in ['question','reference_answer','accepted_aliases','answer_type','category']):
        DIFFS.append({'question_id':qid,'field':'original_vs_combined','json_value':'non-matching', 'xlsx_value':'n/a'})
require(not DIFFS,f'{len(DIFFS)} source mismatches. Stop: do not retroactively alter original experiment.')

for r in EXCEL:
    require(cstrip(r.get('source_name'))!='',f'Source name missing: {r["question_id"]}')
    url=cstrip(r.get('source_reference'))
    p=urllib.parse.urlparse(url)
    require(p.scheme in ('http','https') and p.netloc,f'Invalid source URL: {r["question_id"]}')

print('PASS: all 1250 exact question/reference/aliases/fingerprint comparisons, plus source presence')

In [ ]:
# 5 · Source normalization and transparently *heuristic* institution classification
# A frequent source is not necessarily authoritative or sufficient evidence for a given claim.
PARENTS={
'nasa.gov':'NASA', 'nist.gov':'NIST', 'unesco.org':'UNESCO', 'whc.unesco.org':'UNESCO',
'loc.gov':'Library of Congress','plato.stanford.edu':'Stanford Encyclopedia of Philosophy',
'mathworld.wolfram.com':'Wolfram MathWorld','openstax.org':'OpenStax',
'moma.org':'MoMA','nobelprize.org':'Nobel Prize','tdk.gov.tr':'Turkish Language Association',
'islamansiklopedisi.org.tr':'TDV İslâm Ansiklopedisi',
'esa.int':'European Space Agency','rfc-editor.org':'RFC Editor',
'worldbank.org':'World Bank','un.org':'United Nations','olympics.com':'Olympics',
'britannica.com':'Encyclopaedia Britannica','dictionary.cambridge.org':'Cambridge Dictionary',
'academic.oup.com':'Oxford Academic','genome.gov':'NHGRI', 'nps.gov':'National Park Service',
'who.int':'World Health Organization','iso.org':'ISO','wto.org':'WTO',
'imf.org':'IMF','ncbi.nlm.nih.gov':'NCBI'
}
OFFICIAL={'NASA','NIST','UNESCO','Library of Congress','MoMA','Nobel Prize',
          'Turkish Language Association','European Space Agency','World Bank',
          'United Nations','Olympics','NHGRI','National Park Service',
          'World Health Organization','ISO','WTO','IMF','NCBI'}
SCHOLARLY={'Stanford Encyclopedia of Philosophy','Wolfram MathWorld','OpenStax',
           'Oxford Academic','RFC Editor','TDV İslâm Ansiklopedisi'}
GENERAL={'Encyclopaedia Britannica','Cambridge Dictionary'}

def normalized_host(url):
    host=(urllib.parse.urlparse(url).hostname or '').lower()
    return host[4:] if host.startswith('www.') else host

def organization(host):
    for suffix,name in sorted(PARENTS.items(),key=lambda x:-len(x[0])):
        if host==suffix or host.endswith('.'+suffix):return name
    return host

def source_class(org):
    if org in OFFICIAL:return 'curated_institutional_or_official'
    if org in SCHOLARLY:return 'curated_scholarly_or_technical_reference'
    if org in GENERAL:return 'curated_general_reference'
    return 'other_unclassified'  # not a judgment of unreliability

def split_of(qid):
    if qid in TEST_IDS:return 'FINAL_TEST_250'
    if qid in OLD_IDS:return 'ORIGINAL_DEV_1000'
    if qid in RESERVE_IDS:return 'RESERVE_50'
    raise ValueError(f'Unexpected ID {qid}')

DETAIL=[]
for r in EXCEL:
    qid=r['question_id'];host=normalized_host(cstrip(r['source_reference']))
    org=organization(host)
    original=BY_1250.get(qid,{})
    DETAIL.append({
        'question_id':qid,'dataset_split':split_of(qid),
        'answer_type':r['answer_type'],'category':r['category'],
        'question':r['question'],'reference_answer':r['reference_answer'],
        'source_name':r['source_name'],'source_reference':r['source_reference'],
        'source_host':host,'source_organization':org,'source_class_heuristic':source_class(org),
        'xlsx_verification_status':cstrip(r['verification_status']).lower(),
        'original_json_verification_status':original.get('verification_status','NOT_IN_ORIGINAL_JSON'),
        'validation_notes':cstrip(r.get('validation_notes')),
        'temporal_risk_flag':as_int(r.get('temporal_risk_flag')),
        'ambiguity_risk_flag':as_int(r.get('ambiguity_risk_flag')),
        'validation_decision':cstrip(r.get('validation_decision')),
        'verified_source_text_passage_saved':False,  # XLSX has URL but no quoted supporting passage
    })
COUNTS=collections.Counter((r['dataset_split'],r['xlsx_verification_status']) for r in DETAIL)
print('Status × split:',dict(sorted(COUNTS.items())))

In [ ]:
# 6 · Detailed flags, source rankings, and review queue
SPLITS=['ORIGINAL_DEV_1000','FINAL_TEST_250','RESERVE_50']
SPLIT_STATS=[]
for sp in SPLITS:
    rr=[r for r in DETAIL if r['dataset_split']==sp]
    statuses=collections.Counter(r['xlsx_verification_status'] for r in rr)
    SPLIT_STATS.append({'dataset_split':sp,'n_questions':len(rr),
        'verified_workbook':statuses.get('verified',0),
        'needs_review_workbook':statuses.get('needs review',0),
        'other_status':len(rr)-statuses.get('verified',0)-statuses.get('needs review',0),
        'source_name_present':sum(bool(r['source_name']) for r in rr),
        'source_url_present':sum(bool(r['source_reference']) for r in rr),
        'temporal_risk_flag_count':sum(r['temporal_risk_flag'] for r in rr),
        'ambiguity_risk_flag_count':sum(r['ambiguity_risk_flag'] for r in rr),
        'original_json_verified_flags':sum(r['original_json_verification_status']=='verified' for r in rr)})

USED=[r for r in DETAIL if r['dataset_split']!='RESERVE_50']
def rank_by(field):
    groups=collections.defaultdict(list)
    for r in USED:groups[r[field]].append(r)
    return sorted([{'name':name,'n_questions':len(rs),
      'n_dev1000':sum(r['dataset_split']=='ORIGINAL_DEV_1000' for r in rs),
      'n_test250':sum(r['dataset_split']=='FINAL_TEST_250' for r in rs),
      'n_needs_review':sum(r['xlsx_verification_status']=='needs review' for r in rs),
      'source_class_heuristic':rs[0]['source_class_heuristic']} for name,rs in groups.items()],
      key=lambda z:(-z['n_questions'],z['name']))

HOST_RANK=rank_by('source_host')
ORG_RANK=rank_by('source_organization')
REVIEW_QUEUE=[]
for r in DETAIL:
    if r['xlsx_verification_status']=='needs review' or r['temporal_risk_flag'] or r['ambiguity_risk_flag']:
        v=dict(r)
        v['review_reason']=('; '.join([
            'needs review (requires explicit evidence of second check)' if r['xlsx_verification_status']=='needs review' else '',
            'temporal risk' if r['temporal_risk_flag'] else '',
            'ambiguity risk' if r['ambiguity_risk_flag'] else '']).strip('; '))
        v['claim_resolved_by_file']=False
        REVIEW_QUEUE.append(v)
REVIEW_QUEUE.sort(key=lambda r:({'FINAL_TEST_250':0,'ORIGINAL_DEV_1000':1,'RESERVE_50':2}[r['dataset_split']],r['question_id']))
# Dataset distributions (used only; reserve remains excluded)
def distribution(field):
    out=[]
    for sp in ['ORIGINAL_DEV_1000','FINAL_TEST_250']:
        rr=[r for r in USED if r['dataset_split']==sp]
        counts=collections.Counter(r[field] for r in rr)
        out += [{'dataset_split':sp, field:name, 'n_questions':cnt,
                 'proportion':round(cnt/len(rr),6)} for name,cnt in sorted(counts.items())]
    return out
ANSWER_TYPE_DISTRIBUTION=distribution('answer_type')
CATEGORY_DISTRIBUTION=distribution('category')
SECOND_CHECK_TEMPLATE=[{
    'question_id':r['question_id'],'dataset_split':r['dataset_split'],
    'question':r['question'],'reference_answer':r['reference_answer'],
    'primary_source_url':r['source_reference'],
    'primary_source_name':r['source_name'],
    'flag_type':r['review_reason'],
    'original_review_note':r['validation_notes'],
    'secondary_source_url':'', 'supporting_passage_or_section':'',
    'reviewer':'', 'reviewed_at_utc':'',
    'decision':'',  # acceptable values e.g. CONFIRMED / REVISE / UNRESOLVED
    'resolution_notes':''} for r in REVIEW_QUEUE]
print('Source hosts',len(HOST_RANK),'organizations',len(ORG_RANK))
print('Manual review queue',len(REVIEW_QUEUE),'of which needs_review',sum(r['xlsx_verification_status']=='needs review' for r in REVIEW_QUEUE))
for r in SPLIT_STATS:print(r)

In [ ]:
# 7 · Auditable representative institutional/scholarly source examples
# Human MUST open each link and verify that it directly supports the exact question + answer.
PRIORITY_ORGS=['NASA','UNESCO','NIST','Library of Congress','MoMA','Nobel Prize',
               'Stanford Encyclopedia of Philosophy','Wolfram MathWorld','OpenStax',
               'Turkish Language Association','TDV İslâm Ansiklopedisi','European Space Agency',
               'World Bank','RFC Editor','Oxford Academic']
EXAMPLES=[]; seen_org=set(); seen_id=set()
def add_example(row):
    EXAMPLES.append({'question_id':row['question_id'],'dataset_split':row['dataset_split'],
      'category':row['category'],'answer_type':row['answer_type'],
      'question':row['question'],'reference_answer':row['reference_answer'],
      'source_name':row['source_name'],'source_organization':row['source_organization'],
      'source_reference':row['source_reference'],
      'xlsx_status':row['xlsx_verification_status'],
      'human_passage_check':'PENDING — chosen by metadata only'})
    seen_org.add(row['source_organization']);seen_id.add(row['question_id'])

for split in ['FINAL_TEST_250','ORIGINAL_DEV_1000']:
    target=6
    for org in PRIORITY_ORGS:
        if sum(x['dataset_split']==split for x in EXAMPLES)>=target:break
        candidates=[r for r in USED if r['dataset_split']==split and
                    r['source_organization']==org and r['source_organization'] not in seen_org and
                    r['xlsx_verification_status']=='verified' and not r['validation_notes'] and
                    not r['temporal_risk_flag'] and not r['ambiguity_risk_flag']]
        if candidates: add_example(sorted(candidates,key=lambda r:r['question_id'])[0])
    if sum(x['dataset_split']==split for x in EXAMPLES)<target:
        candidates=[r for r in USED if r['dataset_split']==split and
                    r['source_class_heuristic'] in ('curated_institutional_or_official','curated_scholarly_or_technical_reference') and
                    r['question_id'] not in seen_id and r['xlsx_verification_status']=='verified' and
                    not r['validation_notes'] and not r['temporal_risk_flag'] and not r['ambiguity_risk_flag']]
        for r in sorted(candidates,key=lambda r:r['question_id']):
            if sum(x['dataset_split']==split for x in EXAMPLES)>=target:break
            add_example(r)
require(len(EXAMPLES)==12,'Unable to choose 12 representative examples')
print('Examples chosen:',len(EXAMPLES),'(6 test, 6 original dev; all require human passage checks)')

In [ ]:
# 8 · Immutable CSV/JSON/MD outputs; no network requests; no changes to any dataset.
# Outputs will be rejected rather than overwritten if content differs on a rerun.
OUTPUT.mkdir(parents=True,exist_ok=True)
OUT_HASH={}
def immutable_bytes(path,blob):
    path=Path(path)
    if path.exists():
        require(path.read_bytes()==blob,f'Output already exists with different content: {path}. Use a new revision directory.')
    else:
        tmp=path.with_name(path.name+'.tmp')
        require(not tmp.exists(),f'Unexpected temporary file: {tmp}')
        tmp.write_bytes(blob);os.replace(tmp,path)
    OUT_HASH[path.name]=sha256(path)
def writecsv(name,rows):
    require(rows,f'Cannot write empty CSV {name}')
    buf=io.StringIO(newline='')
    fields=list(rows[0].keys())
    w=csv.DictWriter(buf,fieldnames=fields,extrasaction='ignore',lineterminator='\n')
    w.writeheader();w.writerows(rows)
    immutable_bytes(OUTPUT/name,buf.getvalue().encode('utf-8-sig'))
def writejson(name,obj):
    data=(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,default=str)+'\n').encode('utf-8')
    immutable_bytes(OUTPUT/name,data)

writecsv('stage20_source_crosswalk_v1.csv',DETAIL)
writecsv('stage20_split_status_v1.csv',SPLIT_STATS)
writecsv('stage20_source_host_ranking_v1.csv',HOST_RANK)
writecsv('stage20_source_organization_ranking_v1.csv',ORG_RANK)
writecsv('stage20_review_queue_v1.csv',REVIEW_QUEUE)
writecsv('stage20_representative_sources_v1.csv',EXAMPLES)
writecsv('stage20_answer_type_distribution_v1.csv',ANSWER_TYPE_DISTRIBUTION)
writecsv('stage20_category_distribution_v1.csv',CATEGORY_DISTRIBUTION)
writecsv('stage20_secondary_review_log_template_v1.csv',SECOND_CHECK_TEMPLATE)

summary={'stage':'20_dataset_verification_documentation','status':'PASS_STRUCTURAL_AUDIT',
 'research_character':'post_review_source_documentation','original_data_unchanged':True,
 'test_selection_from_stage13_manifest':True,
 'reviewers_reported_by_researcher':REVIEWERS_REPORTED_BY_RESEARCHER,
 'reviewer_count_independently_documented':False,
 'source_url_is_independent_fact_check':False,
 'verified_source_passages_saved':0,
 'n_excel':len(DETAIL),'n_original_dev':len(OLD_IDS),'n_final_test':len(TEST_IDS),
 'n_reserve':len(RESERVE_IDS),'n_used':len(USED),
 'status_by_split':SPLIT_STATS,
 'n_review_queue':len(REVIEW_QUEUE),
 'n_different_content_or_fingerprints':len(DIFFS),
 'n_org_groups_used':len(ORG_RANK),'n_unique_hosts_used':len(HOST_RANK),
 'n_selected_examples':len(EXAMPLES),'n_review_log_template_rows':len(SECOND_CHECK_TEMPLATE),
 'caveat':'The original JSON marked all 1250 verified; later workbook contains unresolved needs review rows. The source workbook is a post-hoc overlay, not proof of contemporaneous verification or agreement.'}
writejson('stage20_dataset_audit_summary_v1.json',summary)

lines=['# MIDI 2026 — Stage 20 Dataset Verification Audit','',
 '**Result: PASS (structural/source-documentation audit only; NOT independent factual verification).**','',
 f'- Immutable original: {len(OLD_IDS)} previous questions + {len(TEST_IDS)} Stage13 locked final test = {len(COMBINED_IDS)} used questions.',
 f'- Excel review workbook: {len(DETAIL)} sourced rows, including {len(RESERVE_IDS)} excluded reserve questions.',
 f'- Field / fingerprint mismatches in overlapping records: {len(DIFFS)}.',
 f'- URLs present for all {len(DETAIL)} workbook rows; none automatically checked for factual entailment.',
 '', '## Workbook statuses by split', '',
 '| Split | N | verified | needs review | temporal risk | ambiguity risk |',
 '|---|---:|---:|---:|---:|---:|']
for r in SPLIT_STATS:
    lines.append(f"| {r['dataset_split']} | {r['n_questions']} | {r['verified_workbook']} | {r['needs_review_workbook']} | {r['temporal_risk_flag_count']} | {r['ambiguity_risk_flag_count']} |")
lines += ['', '## Frequently used source organizations (used 1,250 only)', '',
 '| Organization | N | Dev | Test | Workbook needs review | Class (heuristic) |',
 '|---|---:|---:|---:|---:|---|']
for r in ORG_RANK[:18]:
    lines.append(f"| {r['name']} | {r['n_questions']} | {r['n_dev1000']} | {r['n_test250']} | {r['n_needs_review']} | {r['source_class_heuristic']} |")
lines += ['', '## Critical methodology cautions', '',
 '1. The earlier published experiment used the immutable original JSON; the source workbook adds/revises provenance after experiment preparation. Do not imply the URLs were recorded or checked at model-inference time unless independently documented.',
 '2. The Excel review flags disagree with the original all-verified status for some used questions. The `needs review` notes must not be silently converted to fully verified because a secondary search was described without a documented link/passage.',
 '3. In particular, there are concerns involving question wording or precision, not merely missing citations. Review all flagged items against primary sources; archive a confirming passage and the human resolution decision.',
 '4. Domain counts are usage statistics only, not content-level correctness, independent annotation, or inter-annotator agreement.',
 '5. The question text, answers, accepted aliases and fingerprint values are unchanged in both source files. Do not modify frozen final test answers to fix review flags without analyzing downstream impact and versioning a new dataset.',
 '6. The 30 development questions used earlier for model selection are still separately excluded by the original feature protocol (Stage16); this documentation split is the original 1000, not the 970 Clean-Dev model-fitting subset.',
 '', '## Researcher-described process (subject to documentary verification)', '',
 'One researcher reports researching candidate questions on public websites, marking those with directly located answers as `verified`, and doing additional research for `needs review` rows. The workbook captures source names/URLs and review notes but not the complete historical review log, excerpts, independent checks or inter-annotator agreement.',
 '', '## For a camera-ready methods paragraph', '',
 'Use **qualified** language such as "Question–answer pairs were manually checked against online references; the revision source register provides per-item URLs and flags potentially ambiguous or weakly supported entries for further review." Only state a total of fully verified items after resolving the review queue.',
 '', '**Representative examples:** `stage20_representative_sources_v1.csv` (metadata-selected; verify cited passages manually before publication).',
 '**Manual verification queue:** `stage20_review_queue_v1.csv`. Fill in supporting passages and decisions in `stage20_secondary_review_log_template_v1.csv` (blank by design).','**Distributions:** `stage20_answer_type_distribution_v1.csv` and `stage20_category_distribution_v1.csv`.']
report='\n'.join(lines)+'\n'
immutable_bytes(OUTPUT/'stage20_report_v1.md',report.encode('utf-8'))

manifest={'stage':'20_dataset_verification_documentation','version':'v1',
 'study_status':'post_review_source_documentation', 'status':'PASS_STRUCTURAL_AUDIT',
 'inputs':{key:{'path':str(INPUTS[key]),'sha256':ACTUAL_SHA[key]} for key in INPUTS},
 'outputs_sha256':dict(sorted(OUT_HASH.items())),
 'source_quality_verified_by_network':False,
 'source_passage_verification_automated':False,
 'modified_previous_stages':False,
 'researcher_described_annotators':REVIEWERS_REPORTED_BY_RESEARCHER,
 'unresolved_review_flags_included':True}
writejson('stage20_evaluation_manifest_v1.json',manifest)
print('\nStage20 DONE: structural audit PASS. Outputs:',OUTPUT)
print('Original test and label files remain unchanged.')
print('IMPORTANT: Resolve manual review queue before claiming all questions are fully verified.')

## Scientific interpretation and next steps

1. Examine `stage20_review_queue_v1.csv` **before publication**. Record each second-check source URL, supporting evidence passage, reviewer decision, and date separately (never overwrite the frozen question set).
2. Review the source examples chosen from institutional and scholarly domains. The notebook selected them by *metadata*, not by opening/entailment-checking the linked page.
3. Describe actual annotation staffing (one person as reported), actual conflict handling (no inter-annotator disagreement process if one person), and the post-hoc timing of source collection transparently.
4. If any questionable **final-test** item has a genuinely incorrect reference answer, do not silently remove/relabel it in the same evaluation. Version any correction separately and state how primary outcomes are affected.
5. Stage 21 paper writing is downstream of this human quality review. This notebook does not fabricate human approval or web verification.